# Lecture 8 Q5: Linear vs Polynomial Regression

Compare how well a linear model and a polynomial model fit nonlinear data.

**Ground-truth function:** y = sin(2π x) + ε,  ε ~ N(0, σ²)

Adjust the noise level and polynomial degree and observe the training MSE for each model.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

rng = np.random.default_rng(42)

def fit_poly(x, y, deg):
    X = np.vander(x, deg + 1, increasing=True)
    theta, *_ = np.linalg.lstsq(X, y, rcond=None)
    return theta

def predict_poly(theta, x):
    X = np.vander(x, len(theta), increasing=True)
    return X @ theta

def mse(y_true, y_pred):
    return float(np.mean((y_true - y_pred) ** 2))

n_slider   = widgets.IntSlider(value=50, min=10, max=200, step=10,
                                description='N (samples)', continuous_update=False,
                                style={'description_width': 'initial'})
sig_slider = widgets.FloatSlider(value=0.1, min=0.0, max=0.5, step=0.05,
                                  description='σ (noise)', continuous_update=False,
                                  style={'description_width': 'initial'})
deg_slider = widgets.IntSlider(value=5, min=1, max=12, step=1,
                                description='Poly degree', continuous_update=False,
                                style={'description_width': 'initial'})

out = widgets.Output()

def render(*_):
    N   = n_slider.value
    sig = sig_slider.value
    deg = deg_slider.value

    x = rng.uniform(0, 1, N)
    y = np.sin(2 * np.pi * x) + rng.normal(0, sig, N)

    theta_lin  = fit_poly(x, y, 1)
    theta_poly = fit_poly(x, y, deg)

    xg = np.linspace(0, 1, 300)
    y_true = np.sin(2 * np.pi * xg)
    y_lin  = predict_poly(theta_lin,  xg)
    y_poly = predict_poly(theta_poly, xg)

    mse_lin  = mse(y, predict_poly(theta_lin,  x))
    mse_poly = mse(y, predict_poly(theta_poly, x))

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Panel 1: Fits
    axes[0].scatter(x, y, s=18, alpha=0.6, color='#475569', label='Data')
    axes[0].plot(xg, y_true, '--', color='#0f766e', lw=1.5, label='True sin(2πx)')
    axes[0].plot(xg, y_lin,  color='#dc2626', lw=2,   label=f'Linear  (MSE={mse_lin:.4f})')
    axes[0].plot(xg, y_poly, color='#2563eb', lw=2,   label=f'Poly deg {deg} (MSE={mse_poly:.4f})')
    axes[0].set_ylim(-2.5, 2.5)
    axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
    axes[0].set_title('Regression fits on nonlinear data')
    axes[0].legend(fontsize=9)

    # Panel 2: MSE bar chart
    bars = axes[1].bar(['Linear', f'Poly deg {deg}'], [mse_lin, mse_poly],
                        color=['#dc2626', '#2563eb'], width=0.4)
    for bar, val in zip(bars, [mse_lin, mse_poly]):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                     f'{val:.4f}', ha='center', va='bottom', fontsize=10)
    axes[1].set_ylabel('Training MSE')
    axes[1].set_title('Training MSE Comparison')
    axes[1].set_ylim(0, max(mse_lin, mse_poly) * 1.3 + 0.01)

    winner = 'Polynomial' if mse_poly < mse_lin else 'Linear'
    fig.suptitle(
        f'{winner} regression achieves lower training MSE  '
        f'(linear={mse_lin:.4f}, poly={mse_poly:.4f})',
        fontsize=10, y=0.02)
    plt.tight_layout()
    with out:
        out.clear_output(wait=True)
        plt.show()

for w in (n_slider, sig_slider, deg_slider):
    w.observe(render, 'value')
display(widgets.HBox([n_slider, sig_slider, deg_slider]), out)
render()